# [MASTERCLASS] Machine Learning Engineering Masterclass: Stages 1 to 4
### *From Problem Scoping to Production-Ready Feature Matrices*
---

### [INSTRUCTOR] Welcome from Your Instructor
Welcome to this hands-on engineering lab! As someone who has deployed machine learning systems at scale, I often tell my students: 
> *"A state-of-the-art model trained on leaky, corrupt, or unscaled data is just an expensive random number generator. Real-world ML is 80% data hygiene, scoping, and rigorous engineering, and 20% algorithm selection."*

You mentioned that you **understand the theoretical concepts** but want to master the **code** so you can implement everything independently. That is the exact right mindset. In this comprehensive notebook, we will translate every single theoretical construct from **Days 1 to 6** into battle-tested, modular Python code.

---

### [ROADMAP] Curriculum Blueprint: Stages 1 to 4

| Stage | Title | Core Objective |
| :--- | :--- | :--- |
| **Stage 1** | **Problem Definition & Scoping** | Translate business pain into ML loss metrics, establish latency/hardware budgets, mask PII, and benchmark baseline heuristics. |
| **Stage 2** | **Data Ingestion & Low-Level Hygiene** | Validate file checksums (SHA-256), enforce schema contracts, route bad records to quarantine (DLQ), and downcast types to save RAM. |
| **Stage 3** | **Data Preprocessing, Cleaning & EDA** | Audit missingness mechanisms (MCAR/MAR/MNAR), standardize structures, cap outliers (IQR/Z-Score), align timezones to UTC, and profile distributions. |
| **Stage 4** | **Feature Engineering, Transformation & Selection** | Build domain ratios, extract cyclical datetime components, encode categories, eliminate multicollinearity, apply Yeo-Johnson & scaling, and select features with zero data leakage. |

---

### [RULE] The Golden Rule of Machine Learning Engineering
**STRICT FIT-TRANSFORM SEPARATION (Zero Data Leakage):**
You must **never** compute statistics (mean, median, standard deviation, category mappings, quantile bounds) on the whole dataset. All transformers must be **fit exclusively on the training partition** and then applied (via `transform`) to the test/production partitions. We will reinforce this pattern at every stage.

Let's begin by importing our foundational toolchain.


In [ ]:
# Fundamental Environment Setup & Toolchain Imports
import os
import sys
import hashlib
import logging
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn Preprocessing & Feature Engineering
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.impute import KNNImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler, PowerTransformer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, mutual_info_classif
from sklearn.decomposition import PCA
import joblib

# Formatting and Visualization Settings
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("MLEngineeringMasterclass")

print("[OK] Environment initialized successfully! Ready for Stage 1.")


---
# ====================================================================
# [STAGE] STAGE 1: Problem Definition & Scoping
# ====================================================================

### [GOAL] Stage Objectives:
1. **Business Objective Translation**: Bridge high-level business goals (e.g. reducing customer churn) into mathematical loss functions ($L_{log}$, RMSE) and quantitative business cost matrices.
2. **Feasibility & Constraint Scoping**: Define real-world operational constraints including inference latency budgets (batch vs real-time API), memory bounds, and regulatory compliance (GDPR / HIPAA PII masking).
3. **Baseline Benchmarking**: Establish a "Zero-Model" baseline heuristic (`DummyClassifier` or domain rules) before writing complex machine learning models.

---

### [THEORY] Theory: Business KPI to ML Metric Mapping
A common beginner mistake is optimizing for simple Accuracy. In unbalanced problems (e.g., 5% churn, 1% fraud), a model that predicts "No Churn" for 100% of customers gets 95% accuracy while bankrupting the business.

We must define an **Asymmetric Business Cost Matrix**:
* **Cost of False Negative ($C_{FN}$)**: A churning customer is predicted as safe $\rightarrow$ Customer leaves $\rightarrow$ **Loss of Customer Lifetime Value (LTV = $500)**.
* **Cost of False Positive ($C_{FP}$)**: A loyal customer is predicted as churning $\rightarrow$ Retention discount coupon offered $\rightarrow$ **Cost of discount voucher ($50)**.

Because $C_{FN} (\$500) \gg C_{FP} (\$50)$, we prioritize **Recall** and penalize False Negatives 10x more heavily than False Positives.


### [STEP] Step 1.1: Business Cost Matrix & Financial Loss Function
Let's translate this financial reality into code by building a custom business loss evaluation function.


In [ ]:
# Step 1.1: Business Impact & Cost Matrix Translation

def compute_business_impact(
    y_true: np.ndarray, 
    y_pred: np.ndarray, 
    cost_fn: float = 500.0,  # Lost customer revenue (False Negative)
    cost_fp: float = 50.0,   # Unnecessary retention discount (False Positive)
    benefit_tp: float = 350.0 # Saved customer revenue minus incentive
) -> Dict[str, float]:
    """
    Calculates the financial business impact of model predictions.
    
    Formula:
      Total Financial Loss = (FN * cost_fn) + (FP * cost_fp) - (TP * benefit_tp)
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    
    total_cost = (fn * cost_fn) + (fp * cost_fp)
    net_saved_revenue = (tp * benefit_tp) - (fp * cost_fp)
    
    return {
        "True Positives (Saved Churners)": int(tp),
        "False Positives (Wasted Discounts)": int(fp),
        "False Negatives (Missed Churners)": int(fn),
        "True Negatives (Correctly Retained)": int(tn),
        "Gross Financial Risk/Loss ($)": float(total_cost),
        "Net Value Generated ($)": float(net_saved_revenue)
    }

# Quick demonstration with simulated predictions
y_actual_demo = np.array([1, 0, 1, 1, 0, 0, 1, 0, 1, 0])
y_dummy_demo  = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0]) # "Predict Nobody Churns"

demo_impact = compute_business_impact(y_actual_demo, y_dummy_demo)
print("--- Step 1.1: Business Cost of 'Predict Nobody Churns' ---")
for k, v in demo_impact.items():
    print(f"{k}: {v}")


### [STEP] Step 1.2: Regulatory Constraints & PII Anonymization (GDPR / HIPAA)
**Production Reality:** Under European GDPR (General Data Protection Regulation) and US HIPAA (Health Insurance Portability and Accountability Act), raw Personally Identifiable Information (PII) such as full names, social security numbers, IP addresses, and customer emails must **never** be piped directly into model training matrices.

We implement cryptographic one-way salted SHA-256 hashing to pseudonymize identifiers while preserving unique record joinability.


In [ ]:
# Step 1.2: Regulatory PII Masking & Cryptographic Pseudonymization

def anonymize_pii_identifier(identifier: str, salt: str = "ML_SALT_SECURE_2026") -> str:
    """
    Converts raw PII string (e.g. Email/Name) into a compliant, irreversible SHA-256 hash.
    """
    if pd.isna(identifier):
        return "UNKNOWN_USER"
    salted_text = f"{identifier.strip().lower()}:{salt}"
    return hashlib.sha256(salted_text.encode("utf-8")).hexdigest()[:16]

# Test PII sanitization
raw_customers = ["aarav.sharma@example.com", "bhavna.patel@corporate.org", "chirag.gupta@domain.in"]
anonymized_ids = [anonymize_pii_identifier(c) for c in raw_customers]

print("--- Step 1.2: PII Anonymization Results ---")
for raw, anon in zip(raw_customers, anonymized_ids):
    print(f"Raw PII: {raw:30} -> Anonymized Token: CUST_{anon}")


### [STEP] Step 1.3: Establishing the Baseline Benchmark (The "Zero-Model" Bar)
Before writing single layers or tree ensembles, we **must establish the baseline**. A standard baseline uses `DummyClassifier` (e.g. predicting the most frequent class or a simple heuristic rule). If your complex ML model cannot substantially outperform this heuristic baseline financially, it should never be shipped to production.


In [ ]:
# Step 1.3: Establishing Baseline Benchmarking

# Synthetic initial evaluation slice
np.random.seed(42)
y_benchmark_true = np.random.choice([0, 1], size=100, p=[0.75, 0.25]) # 25% Churn rate
X_benchmark_dummy = np.zeros((100, 1)) # Placeholder feature matrix

# Baseline 1: Majority Class Predictor (Always Predict 0: No Churn)
dummy_majority = DummyClassifier(strategy="most_frequent")
dummy_majority.fit(X_benchmark_dummy, y_benchmark_true)
y_pred_majority = dummy_majority.predict(X_benchmark_dummy)

# Baseline 2: Stratified Random Guessing (Respecting class proportions)
dummy_stratified = DummyClassifier(strategy="stratified", random_state=42)
dummy_stratified.fit(X_benchmark_dummy, y_benchmark_true)
y_pred_stratified = dummy_stratified.predict(X_benchmark_dummy)

print("--- Step 1.3: Baseline Benchmark Comparison ---")
majority_metrics = compute_business_impact(y_benchmark_true, y_pred_majority)
stratified_metrics = compute_business_impact(y_benchmark_true, y_pred_stratified)

print(f"Majority Rule Baseline Net Loss : ${majority_metrics['Gross Financial Risk/Loss ($)']:.2f}")
print(f"Stratified Baseline Net Loss   : ${stratified_metrics['Gross Financial Risk/Loss ($)']:.2f}")
print("-> Any machine learning model we build MUST reduce this loss substantially to justify deployment.")


---
## [CHECKPOINT] STAGE 1 CHECKPOINT & TRANSITION
> [!NOTE]
> **Stage 1 Summary:** We formulated the business problem, mapped financial costs directly to classification errors, established PII privacy safeguards, and measured the minimum heuristic baseline to beat.
> 
> **Next Up:** Now that the problem is defined, we move to **Stage 2: Data Ingestion & Low-Level Hygiene**. We will validate file integrity, enforce schema contracts, route bad records to quarantine, and downcast data types to optimize memory.
---


---
# ====================================================================
# [STAGE] STAGE 2: Data Ingestion & Low-Level Hygiene
# ====================================================================

### [GOAL] Stage Objectives:
1. **File Integrity & Zero-Byte Checks**: Ensure incoming files exist, are non-empty, and verify cryptographic SHA-256 checksums to detect file truncation or corruption.
2. **Schema Contract Enforcement**: Enforce mandatory column presence, strict data types, nullability boundaries, and allowed category values.
3. **Dead-Letter Queue (Quarantine Routing)**: Intercept corrupted or out-of-bounds records and isolate them into a quarantine directory so corrupted records never crash downstream pipelines.
4. **Low-Level Memory Optimization**: Downcast numerical data types (`int64` $\rightarrow$ `int8/16/32`, `float64` $\rightarrow$ `float32`) to reduce the RAM footprint by 50-75%.
5. **Out-of-Core Processing**: Demonstrate chunked file reading to safely process multi-gigabyte files within modest RAM limits.

Let's begin by generating a realistic raw payload with intentional dirty edge cases.


### [STEP] Step 2.1: File Creation, Zero-Byte Check & SHA-256 Checksum Validation
We write a defensive file integrity validator that inspects raw payloads before loading them into memory.


In [ ]:
# Step 2.1: File Integrity, Zero-Byte Check & SHA-256 Checksum Validator

# 1. Setup Data Directory Structure
raw_data_dir = Path("data/raw")
quarantine_data_dir = Path("data/quarantine")
processed_data_dir = Path("data/processed")

raw_data_dir.mkdir(parents=True, exist_ok=True)
quarantine_data_dir.mkdir(parents=True, exist_ok=True)
processed_data_dir.mkdir(parents=True, exist_ok=True)

raw_telemetry_file = raw_data_dir / "customer_subscription_telemetry.csv"

# 2. Generate Realistic Raw CSV Dataset with deliberate dirty records:
raw_csv_content = """customer_id,tenure_months,monthly_charges,total_charges,contract_type,payment_method,timestamp,churn_status
CUST_001,12,29.95,359.4,month-to-month,electronic_check,2024-01-15 08:30:00,no
CUST_002,24,75.50,1812.0,one-year,mailed_check,2024-01-15 09:15:00,yes
CUST_003,-5,105.00,525.0,two-year,credit_card,2024-01-15 10:00:00,no
CUST_004,36,65.00,,one-year,bank_transfer,2024-01-15 11:20:00,no
CUST_005,6,45.20,-999.0,month-to-month,crypto_token,2024-01-15 12:45:00,yes
CUST_006,48,89.50,4296.0,two-year,electronic_check,2024-01-15 14:10:00,no
CUST_007,18,55.00,990.0,month-to-month,mailed_check,2024-01-15 15:00:00,yes
CUST_008,60,115.00,6900.0,two-year,credit_card,2024-01-15 16:30:00,no
CUST_009,2,20.00,40.0,month-to-month,electronic_check,2024-01-15 17:45:00,yes
CUST_010,72,120.00,8640.0,two-year,bank_transfer,2024-01-15 18:20:00,no
CUST_011,15,35.00,525.0,one-year,credit_card,2024-01-15 19:10:00,no
CUST_012,3,60.00,180.0,month-to-month,electronic_check,2024-01-15 20:00:00,yes
"""

raw_telemetry_file.write_text(raw_csv_content.strip(), encoding="utf-8")
print(f"Created sample telemetry file: {raw_telemetry_file}")
e
# 3. Defensive Checksum & Zero-Byte Verification Function
def calculate_sha256(file_path: Path, chunk_size: int = 65536) -> str:
    """Calculate SHA-256 hash using chunked binary streaming to prevent memory blowup."""
    hasher = hashlib.sha256()
    with open(file_path, "rb") as f:
        while chunk := f.read(chunk_size):
            hasher.update(chunk)
    return hasher.hexdigest()

def validate_file_integrity(file_path: Path, expected_sha256: Optional[str] = None) -> bool:
    """Validates existence, zero-byte status, and optional hash matches."""
    if not file_path.exists():
        logger.error(f"[ERROR] File does not exist: {file_path}")
        return False
    
    file_size = file_path.stat().st_size
    if file_size == 0:
        logger.critical(f"[ERROR] Zero-byte failure: {file_path} is completely empty (0 bytes)!")
        return False
    
    computed_hash = calculate_sha256(file_path)
    logger.info(f"[OK] File Size: {file_size} bytes | SHA-256: {computed_hash}")
    
    if expected_sha256 and computed_hash != expected_sha256:
        logger.critical(f"[ERROR] Hash mismatch! Expected {expected_sha256}, got {computed_hash}")
        return False
        
    return True

is_healthy = validate_file_integrity(raw_telemetry_file)
print(f"File integrity status: {is_healthy}")


### [STEP] Step 2.2: Schema Contract Enforcement & Dead-Letter Queue (DLQ)
In production pipelines, upstream engineers or third-party APIs can change schema formats without notice. If invalid records leak into feature transformers, they crash the model. 

Here we enforce a **strict data contract**:
* `tenure_months` must be between $0$ and $120$ (intercepts `CUST_003` with tenure `-5`).
* `payment_method` must be in our allowed set (intercepts `CUST_005` with unauthorized `crypto_token`).
* Corrupted records are partitioned directly to a **Quarantine CSV (DLQ)** with diagnostic failure reasons, allowing healthy records to proceed.


In [ ]:
# Step 2.2: Schema Contract Enforcement & Quarantine Isolation

# Define Data Contract Rules
SCHEMA_CONTRACT = {
    "expected_columns": {
        "customer_id": {"type": "string", "nullable": False},
        "tenure_months": {"type": "int64", "min_val": 0, "max_val": 120, "nullable": False},
        "monthly_charges": {"type": "float64", "min_val": 0.0, "max_val": 2000.0, "nullable": False},
        "total_charges": {"type": "float64", "min_val": 0.0, "max_val": 100000.0, "nullable": True},
        "contract_type": {"type": "string", "allowed_values": ["month-to-month", "one-year", "two-year"], "nullable": False},
        "payment_method": {"type": "string", "allowed_values": ["electronic_check", "mailed_check", "bank_transfer", "credit_card"], "nullable": False},
        "churn_status": {"type": "string", "allowed_values": ["yes", "no"], "nullable": False}
    }
}

def enforce_schema_and_quarantine(
    df: pd.DataFrame, 
    contract: dict, 
    quarantine_output: Path
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Validates records against contract rules.
    Partitions data into (df_valid, df_quarantined).
    """
    valid_mask = np.ones(len(df), dtype=bool)
    rejection_reasons = [[] for _ in range(len(df))]
    rules = contract["expected_columns"]
    
    # 1. Schema Drift Check (Missing required columns)
    missing_cols = [col for col in rules.keys() if col not in df.columns]
    if missing_cols:
        raise ValueError(f"FATAL SCHEMA DRIFT: Missing required columns {missing_cols}")

    # 2. Row-by-Row Contract Validation
    for i, row in df.iterrows():
        # Tenure Boundary
        if row["tenure_months"] < rules["tenure_months"]["min_val"]:
            valid_mask[i] = False
            rejection_reasons[i].append(f"tenure_months ({row['tenure_months']}) < min 0")
            
        # Monthly Charges Boundary
        if row["monthly_charges"] < rules["monthly_charges"]["min_val"]:
            valid_mask[i] = False
            rejection_reasons[i].append(f"monthly_charges ({row['monthly_charges']}) < min 0.0")
            
        # Allowed Categorical Values
        if row["payment_method"] not in rules["payment_method"]["allowed_values"]:
            valid_mask[i] = False
            rejection_reasons[i].append(f"payment_method '{row['payment_method']}' not permitted")
            
    df_valid = df[valid_mask].copy().reset_index(drop=True)
    df_quarantine = df[~valid_mask].copy()
    df_quarantine["quarantine_reason"] = ["; ".join(reasons) for reasons, keep in zip(rejection_reasons, valid_mask) if not keep]
    
    if not df_quarantine.empty:
        df_quarantine.to_csv(quarantine_output, index=False)
        logger.warning(f"[WARNING] Quarantined {len(df_quarantine)} invalid records -> {quarantine_output}")
        
    logger.info(f"[OK] Valid records retained: {len(df_valid)} out of {len(df)} initial rows.")
    return df_valid, df_quarantine

# Execute schema enforcement
df_raw_loaded = pd.read_csv(raw_telemetry_file)
df_valid_stage2, df_quarantined = enforce_schema_and_quarantine(
    df_raw_loaded, 
    SCHEMA_CONTRACT, 
    quarantine_data_dir / "quarantined_telemetry.csv"
)

print("\n--- Quarantined Records Preview ---")
print(df_quarantined[["customer_id", "tenure_months", "payment_method", "quarantine_reason"]])


### [STEP] Step 2.3: Low-Level Memory Optimization & Numerical Type Downcasting
**Production Context:** By default, pandas loads all integer numbers as `int64` (8 bytes per value) and all floating-point numbers as `float64` (8 bytes per value). In enterprise datasets with millions of rows, this exhausts available RAM and causes out-of-memory (`OOMKilled`) termination.

We inspect the actual minimum and maximum bounds of our data:
* An integer like `tenure_months` ranges from $0$ to $72$. The `int8` data type holds numbers from $-128$ to $+127$ using only **1 byte** instead of 8!
* Floating-point figures like charges can be safely represented by `float32` (4 bytes) with negligible precision loss.

Let's implement automated downcasting.


In [ ]:
# Step 2.3: Memory Optimization & Numerical Downcasting

def downcast_numeric_types(df: pd.DataFrame) -> Tuple[pd.DataFrame, float, float]:
    """
    Aggressively downcasts int64 and float64 columns to their minimal lossless representation.
    Returns (optimized_df, initial_mb, optimized_mb).
    """
    df_opt = df.copy()
    start_bytes = df_opt.memory_usage(deep=True).sum()
    start_mb = start_bytes / (1024 * 1024)
    
    for col in df_opt.select_dtypes(include=["integer"]).columns:
        c_min = df_opt[col].min()
        c_max = df_opt[col].max()
        
        if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
            df_opt[col] = df_opt[col].astype(np.int8)
        elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
            df_opt[col] = df_opt[col].astype(np.int16)
        elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
            df_opt[col] = df_opt[col].astype(np.int32)
            
    for col in df_opt.select_dtypes(include=["floating"]).columns:
        c_min = df_opt[col].min()
        c_max = df_opt[col].max()
        
        if c_min >= np.finfo(np.float32).min and c_max <= np.finfo(np.float32).max:
            df_opt[col] = df_opt[col].astype(np.float32)
            
    end_bytes = df_opt.memory_usage(deep=True).sum()
    end_mb = end_bytes / (1024 * 1024)
    
    return df_opt, start_mb, end_mb

df_downcasted, initial_mb, final_mb = downcast_numeric_types(df_valid_stage2)
print("--- Step 2.3: Memory Downcasting Results ---")
print(f"Memory Before Downcasting : {initial_mb * 1024:.2f} KB")
print(f"Memory After Downcasting  : {final_mb * 1024:.2f} KB")
print(f"Dtypes Optimized:\n{df_downcasted.dtypes}")


### [STEP] Step 2.4: Out-of-Core & Chunked Ingestion Pattern
When dataset sizes exceed available RAM (e.g. a 20GB CSV on an 8GB machine), we cannot call `pd.read_csv()` in one go. We utilize **chunked iteration** via `chunksize` to stream batches sequentially.


In [ ]:
# Step 2.4: Out-of-Core Batch Ingestion via Chunksize

def process_file_in_chunks(file_path: Path, chunk_size: int = 4) -> pd.DataFrame:
    """
    Demonstrates out-of-core chunked streaming.
    Processes stream chunk-by-chunk without loading whole payload into memory.
    """
    processed_chunks = []
    chunk_iterator = pd.read_csv(file_path, chunksize=chunk_size)
    
    for chunk_idx, chunk in enumerate(chunk_iterator):
        logger.info(f"Ingesting stream Chunk #{chunk_idx + 1} ({len(chunk)} rows)...")
        # Apply low-level filtering / downcasting per chunk
        chunk_clean, _, _ = downcast_numeric_types(chunk)
        processed_chunks.append(chunk_clean)
        
    full_df = pd.concat(processed_chunks, axis=0).reset_index(drop=True)
    return full_df

df_chunked_demo = process_file_in_chunks(raw_telemetry_file, chunk_size=4)
print(f"Successfully processed {len(df_chunked_demo)} rows using chunked streaming.")


---
## [CHECKPOINT] STAGE 2 CHECKPOINT & TRANSITION
> [!NOTE]
> **Stage 2 Summary:** We validated file integrity with SHA-256, intercepted invalid records into quarantine via schema contracts, downcasted numerical types to conserve RAM, and established out-of-core chunked ingestion patterns.
> 
> **Next Up:** Now we enter **Stage 3: Data Preprocessing, Cleaning & EDA ("Dirty Data" Audit)**. We will diagnose missingness mechanisms, clean structural errors, detect and cap outliers, handle temporal alignment, and profile feature distributions.
---


---
# ====================================================================
# [STAGE] STAGE 3: Data Preprocessing, Cleaning & EDA
# ====================================================================

### [GOAL] Stage Objectives:
1. **Missing Data Mechanism Analysis (MCAR vs MAR vs MNAR)**: Distinguish random missingness from systematic missingness, mask sentinel codes (e.g. `-999.0` $\rightarrow$ `NaN`), create binary missingness indicators (`_is_missing`), and apply multivariate KNN Imputation.
2. **Structural & Text Hygiene**: Enforce strict `snake_case` column conventions, strip whitespace, normalize text casing, remove duplicate records, and resolve categorical typos using Levenshtein distance string matching (`rapidfuzz`).
3. **Statistical Outlier Detection & Capping**: Apply Tukey's Interquartile Range (IQR) fence ($1.5 \times IQR$) and Z-Score bounds ($|z| > 3.0$), capping extreme values to preserve statistical sample size.
4. **Temporal Alignment & Look-Ahead Bias Prevention**: Localize datetimes to UTC to prevent silent join discrepancies, and resample irregular series with forward-fill (`ffill`) to prevent look-ahead leakage.
5. **Mathematical Sanitization**: Intercept IEEE 754 division-by-zero, mask `inf`/`-inf`, and handle float precision rounding errors.
6. **Exploratory Data Analysis (EDA)**: Profile univariate distributions, compute bivariate correlation matrices, and analyze feature-to-target interactions.

Let's work through each step sequentially.


### [STEP] Step 3.1: Missingness Mechanism Audit, Sentinel Masking & Indicator Flagging
#### [THEORY] Theory on Missingness Mechanisms:
* **MCAR (Missing Completely at Random)**: Missingness is purely random noise (e.g., hardware sensor glitch). Mean/median imputation does not bias estimates.
* **MAR (Missing at Random)**: Missingness depends on observed features (e.g., younger users rarely report income).
* **MNAR (Missing Not at Random)**: The value is missing *because of what the value is* (e.g., customers with very high debt refuse to report it, or systems enter a sentinel code like `-999.0` to denote missingness).

**Golden Rule:** Always preserve the "signal of absence" by adding a binary flag column (`col_is_missing = 1/0`) before imputing!


In [ ]:
# Step 3.1: Missingness Analysis, Sentinel Masking & Multivariate KNN Imputation

# Enrich our working dataframe with intentional missing values and MNAR sentinel codes
df_stage3 = df_downcasted.copy()
# Insert an MNAR sentinel in row 3
df_stage3.loc[3, "total_charges"] = np.nan
df_stage3.loc[1, "monthly_charges"] = -999.0  # Sentinel code indicating missingness

# 1. Audit Missingness
def audit_missingness(df: pd.DataFrame) -> pd.DataFrame:
    """Generates a comprehensive missingness audit report."""
    counts = df.isnull().sum()
    pcts = (counts / len(df)) * 100
    report = pd.DataFrame({"missing_count": counts, "missing_percent": pcts})
    return report[report["missing_count"] > 0].sort_values(by="missing_percent", ascending=False)

print("--- Initial Missingness Audit ---")
print(audit_missingness(df_stage3))

# 2. Sentinel Code Masking
def mask_sentinel_codes(df: pd.DataFrame, sentinel_val: float = -999.0) -> pd.DataFrame:
    """Replaces legacy sentinel codes with standard IEEE NaN."""
    df_clean = df.copy()
    num_cols = df_clean.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        sentinel_count = (df_clean[col] == sentinel_val).sum()
        if sentinel_count > 0:
            df_clean[col] = df_clean[col].replace(sentinel_val, np.nan)
            logger.info(f"Masked {sentinel_count} sentinel ({sentinel_val}) occurrences in '{col}' to NaN.")
    return df_clean

df_stage3 = mask_sentinel_codes(df_stage3, sentinel_val=-999.0)

# 3. Missingness Indicator Flagging (Preserving Signal of Absence)
def add_missingness_indicators(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    """Creates explicit binary flags: 1 if missing, 0 otherwise."""
    df_clean = df.copy()
    for col in cols:
        if col in df_clean.columns:
            indicator_col = f"{col}_is_missing"
            df_clean[indicator_col] = df_clean[col].isna().astype(np.int8)
            logger.info(f"Constructed missingness indicator feature: '{indicator_col}'")
    return df_clean

df_stage3 = add_missingness_indicators(df_stage3, ["total_charges", "monthly_charges"])
# df_stage3 
# 4. Multivariate KNN Imputation for Continuous Numerical Columns
def impute_numerical_knn(df: pd.DataFrame, num_cols: List[str], n_neighbors: int = 3) -> pd.DataFrame:
    """Applies multivariate KNN imputer to continuous features based on feature similarity."""
    df_clean = df.copy()
    imputer = KNNImputer(n_neighbors=n_neighbors, weights="uniform")
    df_clean[num_cols] = imputer.fit_transform(df_clean[num_cols])
    return df_clean

num_features = ["tenure_months", "monthly_charges", "total_charges"]
df_stage3 = impute_numerical_knn(df_stage3, num_features, n_neighbors=2)

print("\n--- Post-Imputation Missingness Audit ---")
print(audit_missingness(df_stage3))
print("[OK] All missing continuous values imputed while preserving missingness signals!")


### [STEP] Step 3.2: Structural Hygiene, Text Normalization & Fuzzy String Matching
In real-world data, human entries or diverse upstream forms create dirty text:
* Inconsistent casing: `"Month-to-Month"`, `"month to month"`, `"m2m"`.
* Dirty whitespaces: `"  mailed_check  "`.
* Exact duplicate rows: Same user re-sent through pipeline.

We standardize column headers to `snake_case`, normalize strings, drop duplicate rows, and apply **Levenshtein distance string matching** to consolidate fuzzy variants into canonical categories.


In [ ]:
# Step 3.2: Structural Hygiene, Deduplication & Levenshtein Matching

# 1. Standardize Column Headers to snake_case
def standardize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Converts column names to trimmed, lowercased snake_case."""
    df_clean = df.copy()
    df_clean.columns = (
        df_clean.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )
    return df_clean

# 2. Text Normalization (Trimming whitespace and lowercasing)
def normalize_text_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Strips leading/trailing whitespace and normalizes case for string columns."""
    df_clean = df.copy()
    str_cols = df_clean.select_dtypes(include=["object", "string"]).columns
    for col in str_cols:
        df_clean[col] = df_clean[col].astype(str).str.strip().str.lower()
    return df_clean

# 3. Deduplication
def remove_duplicate_records(df: pd.DataFrame, subset_cols: Optional[List[str]] = None) -> pd.DataFrame:
    """Removes exact duplicate rows."""
    initial_len = len(df)
    df_clean = df.drop_duplicates(subset=subset_cols).reset_index(drop=True)
    dropped = initial_len - len(df_clean)
    if dropped > 0:
        logger.info(f"Removed {dropped} duplicate records.")
    return df_clean

# 4. Levenshtein Distance Fuzzy String Matching (Pure Python fallback + Rapidfuzz)
def compute_levenshtein_ratio(s1: str, s2: str) -> float:
    """Computes similarity ratio between 0.0 and 1.0 using Levenshtein distance matrix."""
    if s1 == s2: return 1.0
    rows = len(s1) + 1
    cols = len(s2) + 1
    dist = [[0 for _ in range(cols)] for _ in range(rows)]
    for i in range(1, rows): dist[i][0] = i
    for j in range(1, cols): dist[0][j] = j
    for col in range(1, cols):
        for row in range(1, rows):
            cost = 0 if s1[row - 1] == s2[col - 1] else 1
            dist[row][col] = min(dist[row - 1][col] + 1, dist[row][col - 1] + 1, dist[row - 1][col - 1] + cost)
    max_len = max(len(s1), len(s2))
    return 1.0 - (dist[-1][-1] / max_len)

def resolve_categorical_typos(
    df: pd.DataFrame, 
    col: str, 
    canonical_targets: List[str], 
    threshold: float = 0.75
) -> pd.DataFrame:
    """Maps messy variants to canonical category names based on string similarity."""
    df_clean = df.copy()
    def map_val(val):
        for target in canonical_targets:
            if compute_levenshtein_ratio(str(val), target) >= threshold:
                return target
        return val
    df_clean[col] = df_clean[col].apply(map_val)
    return df_clean

# Execute structural hygiene
df_stage3 = standardize_column_names(df_stage3)
df_stage3 = normalize_text_columns(df_stage3)
df_stage3 = remove_duplicate_records(df_stage3, subset_cols=["customer_id"])
df_stage3 = resolve_categorical_typos(df_stage3, "contract_type", ["month-to-month", "one-year", "two-year"])

print("--- Step 3.2: Cleaned Data Structure Sample ---")
print(df_stage3[["customer_id", "contract_type", "payment_method"]].head(5))


### [STEP] Step 3.3: Statistical Outlier Detection & Capping (IQR & Z-Score)
#### [THEORY] Capping (Winsorization) vs Row Deletion:
Dropping rows with extreme values drops valuable signal and creates sample-selection bias. Instead, in production feature pipelines, we **cap (Winsorize)** continuous variables to statistical fences:
* **Tukey's IQR Fence**:
  $$IQR = Q_3 - Q_1$$
  $$\text{Lower Fence} = Q_1 - 1.5 \times IQR, \quad \text{Upper Fence} = Q_3 + 1.5 \times IQR$$
* **Z-Score Fence**:
  $$\text{Bound} = \mu \pm 3.0 \times \sigma$$


In [ ]:
# Step 3.3: Outlier Capping via Tukey's IQR and Z-Score

def cap_outliers_iqr(df: pd.DataFrame, num_cols: List[str], multiplier: float = 1.5) -> pd.DataFrame:
    """Clips extreme continuous values outside Tukey's IQR fence."""
    df_clean = df.copy()
    for col in num_cols:
        q1 = df_clean[col].quantile(0.25)
        q3 = df_clean[col].quantile(0.75)
        iqr = q3 - q1
        lower_fence = q1 - (multiplier * iqr)
        upper_fence = q3 + (multiplier * iqr)
        
        outliers_detected = ((df_clean[col] < lower_fence) | (df_clean[col] > upper_fence)).sum()
        if outliers_detected > 0:
            df_clean[col] = df_clean[col].clip(lower=lower_fence, upper=upper_fence)
            logger.info(f"Capped {outliers_detected} IQR outliers in '{col}' to [{lower_fence:.2f}, {upper_fence:.2f}].")
    return df_clean

def cap_outliers_zscore(df: pd.DataFrame, num_cols: List[str], threshold: float = 3.0) -> pd.DataFrame:
    """Clips extreme continuous values outside Z-Score bounds."""
    df_clean = df.copy()
    for col in num_cols:
        mean_val = df_clean[col].mean()
        std_val = df_clean[col].std()
        if std_val == 0 or pd.isna(std_val):
            continue
        lower_bound = mean_val - (threshold * std_val)
        upper_bound = mean_val + (threshold * std_val)
        df_clean[col] = df_clean[col].clip(lower=lower_bound, upper=upper_bound)
    return df_clean

# Apply IQR capping to financial telemetry
df_stage3 = cap_outliers_iqr(df_stage3, ["monthly_charges", "total_charges"])
print("[OK] Outlier capping completed successfully.")


### [STEP] Step 3.4: Temporal Alignment & Resampling without Look-Ahead Bias
#### [THEORY] Timezone Integrity & Look-Ahead Bias:
* **Timezone Discrepancy**: If timestamps are collected across different global regions with naive timezones, merging them causes silent alignment errors. We strictly localize to **UTC**.
* **Look-Ahead Bias**: In time-series resampling, backward-filling (`bfill`) uses future values to impute past points. This is severe **data leakage**! We use forward-fill (`ffill`), ensuring only historically available observations are propagated.


In [ ]:
# Step 3.4: Timezone Normalization & Forward-Fill Resampling

def normalize_datetime_to_utc(df: pd.DataFrame, dt_col: str, target_tz: str = "UTC") -> pd.DataFrame:
    """Converts timezone-naive or aware timestamps strictly to standard UTC."""
    df_clean = df.copy()
    if dt_col in df_clean.columns:
        dt_series = pd.to_datetime(df_clean[dt_col], errors="coerce")
        if dt_series.dt.tz is None:
            # Naive datetime: localize to target UTC
            dt_series = dt_series.dt.tz_localize(target_tz, ambiguous="NaT", nonexistent="shift_forward")
        else:
            # Aware datetime: convert directly to target UTC
            dt_series = dt_series.dt.tz_convert(target_tz)
        df_clean[dt_col] = dt_series
        logger.info(f"Successfully normalized '{dt_col}' to {target_tz}.")
    return df_clean

df_stage3 = normalize_datetime_to_utc(df_stage3, "timestamp", target_tz="UTC")

# Demonstrate look-ahead-safe resampling on irregular telemetry
def resample_time_series_safe(df: pd.DataFrame, time_col: str, val_col: str, freq: str = "1h") -> pd.DataFrame:
    """Resamples series using forward-fill (ffill) to eliminate look-ahead leakage."""
    ts_df = df[[time_col, val_col]].dropna().copy()
    ts_df = ts_df.sort_values(by=time_col).set_index(time_col)
    resampled = ts_df.resample(freq).mean().ffill() # strictly forward-fill past values
    return resampled.reset_index()

resampled_demo = resample_time_series_safe(df_stage3, "timestamp", "monthly_charges", freq="2h")
print("--- Step 3.4: Look-Ahead Safe Resampled Telemetry (Head) ---")
print(resampled_demo.head(4))


### [STEP] Step 3.5: Mathematical Sanitization & IEEE 754 Floating-Point Safeguards
When building interaction ratios (e.g. dividing by zero or nearly zero values), NumPy generates IEEE 754 `np.inf`, `-np.inf`, or `NaN`. Feeding `inf` to linear algebra solvers in Scikit-Learn or PyTorch causes immediate `ValueError: Input contains NaN, infinity or a value too large for dtype('float64')`.

We write vector-safe division and infinity masking functions.


In [ ]:
# Step 3.5: Mathematical Sanitization & Division-by-Zero Interceptor

def safe_divide(numerator: pd.Series, denominator: pd.Series, default_val: float = 0.0) -> pd.Series:
    """Performs division while safely intercepting zero denominators and infinities."""
    denom_clean = denominator.replace(0, np.nan)
    result = numerator / denom_clean
    return result.replace([np.inf, -np.inf], np.nan).fillna(default_val)

def sanitize_floating_point_matrix(df: pd.DataFrame) -> pd.DataFrame:
    """Scans all numerical columns, masking IEEE 754 infinities with finite fallbacks."""
    df_clean = df.copy()
    num_cols = df_clean.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        inf_mask = np.isinf(df_clean[col])
        if inf_mask.sum() > 0:
            df_clean[col] = df_clean[col].replace([np.inf, -np.inf], np.nan)
            logger.warning(f"Sanitized {inf_mask.sum()} infinities in '{col}' to NaN.")
    return df_clean

df_stage3 = sanitize_floating_point_matrix(df_stage3)
print("[OK] Mathematical sanitization complete.")


### [STEP] Step 3.6: Exploratory Data Analysis (EDA) & Target Profiling
EDA is how we discover patterns and validate assumptions before creating features.
We inspect:
1. **Univariate distributions**: Min, max, median, skewness.
2. **Correlation matrix**: High collinearity indicators.
3. **Target interaction**: Churn distribution across categorical features.


In [ ]:
# Step 3.6: Automated Exploratory Data Analysis (EDA) Profiling

# Map churn target to binary numeric (1 = Churn 'yes', 0 = Retained 'no')
df_stage3["churn_target"] = (df_stage3["churn_status"] == "yes").astype(np.int8)

print("--- Step 3.6: Univariate Feature Statistics & Skewness ---")
numeric_eda = df_stage3.select_dtypes(include=[np.number])
print(numeric_eda.describe().T[["mean", "std", "min", "50%", "max"]])
print("\nFeature Skewness:\n", numeric_eda.skew().round(2))

print("\n--- Bivariate Target Interaction Analysis ---")
for cat_col in ["contract_type", "payment_method"]:
    grouped = df_stage3.groupby(cat_col)["churn_target"].agg(["count", "mean"])
    grouped.columns = ["Customer Count", "Churn Rate"]
    print(f"\nChurn Breakdown by {cat_col}:\n{grouped.round(2)}")


---
## [CHECKPOINT] STAGE 3 CHECKPOINT & TRANSITION
> [!NOTE]
> **Stage 3 Summary:** We resolved missingness mechanisms (with indicator flagging & KNN imputation), cleaned text and schema, capped outliers with IQR fences, aligned datetimes to UTC, masked infinities, and performed exploratory profiling.
> 
> **Next Up:** Now we enter **Stage 4: Feature Engineering, Transformation & Selection**. We will construct domain interaction features, cyclical trigonometric features, encode categories, remove quasi-constants and multicollinearity, apply Yeo-Johnson transformations and scaling, and select optimal features.
---


---
# ====================================================================
# [STAGE] STAGE 4: Feature Engineering, Transformation & Selection
# ====================================================================

### [GOAL] Stage Objectives:
1. **Preventing Data Leakage**: Establish a strict Train/Test split BEFORE fitting any transformer.
2. **Domain-Specific Feature Engineering**: Construct domain interaction ratios (`charge_per_tenure_month`, `monthly_to_total_ratio`).
3. **Temporal Feature Decomposition**: Extract calendar sub-components and transform periodic intervals (months) into continuous cyclical Sine/Cosine coordinates.
4. **Categorical Encoding & Rare Label Binning**: Group long-tail infrequent categories into `'Other'`, and apply One-Hot Encoding (`handle_unknown='ignore'`) for low-cardinality features.
5. **Low-Variance & Quasi-Constant Elimination**: Drop features offering zero predictive variance using `VarianceThreshold`.
6. **Multicollinearity Removal**: Detect redundant feature pairs with Pearson $|r| > 0.85$ and drop collinear features.
7. **Mathematical Power Transformations & Scaling**: Normalize skewed distributions with Yeo-Johnson (`PowerTransformer`) and scale features with `StandardScaler`.
8. **Automated Feature Selection & PCA**: Select top features via ANOVA F-classif (`SelectKBest`) and project dimensions via Principal Component Analysis (PCA).
9. **Artifact Serialization**: Persist all fitted transformers (`joblib`) for production deployment.

Let's step through every operation with strict fit-transform separation.


### [STEP] Step 4.1: Stratified Train-Test Split (The Data Leakage Firewall)
#### [THEORY] The Prime Directive of Machine Learning:
Every statistical transformer (StandardScaler, Imputer, VarianceThreshold, OneHotEncoder) calculates internal statistics (e.g., $\mu, \sigma$, category levels). If you calculate these on the whole dataset before splitting, **test set distribution information leaks into the training pipeline**. The model will appear to perform well in cross-validation but will degrade in production.

We split first, fit exclusively on `train`, and merely transform `test`.


In [ ]:
# Step 4.1: Stratified Train/Test Split (Preventing Data Leakage)

# Expand dataset slice slightly to demonstrate multi-feature selection cleanly
np.random.seed(42)
extra_records = []
for i in range(13, 61):
    extra_records.append({
        "customer_id": f"CUST_{i:03d}",
        "tenure_months": np.random.randint(1, 72),
        "monthly_charges": float(np.random.uniform(20.0, 110.0)),
        "total_charges": float(np.random.uniform(100.0, 7500.0)),
        "contract_type": np.random.choice(["month-to-month", "one-year", "two-year"], p=[0.5, 0.3, 0.2]),
        "payment_method": np.random.choice(["electronic_check", "mailed_check", "credit_card", "bank_transfer"], p=[0.4, 0.2, 0.2, 0.2]),
        "timestamp": pd.Timestamp("2024-01-15 12:00:00", tz="UTC") + pd.Timedelta(hours=i),
        "churn_status": np.random.choice(["yes", "no"], p=[0.3, 0.7]),
        "total_charges_is_missing": 0,
        "monthly_charges_is_missing": 0,
        "churn_target": 0
    })
extra_df = pd.DataFrame(extra_records)
extra_df["churn_target"] = (extra_df["churn_status"] == "yes").astype(np.int8)

df_full = pd.concat([df_stage3, extra_df], ignore_index=True)

# Separate Features (X) and Target (y)
X_raw = df_full.drop(columns=["churn_status", "churn_target"])
y_raw = df_full["churn_target"]

# Stratified Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, 
    test_size=0.25, 
    stratify=y_raw, 
    random_state=42
)

print(f"--- Step 4.1: Data Split Complete ---")
print(f"X_train Shape: {X_train.shape} | y_train Churn Rate: {y_train.mean():.2%}")
print(f"X_test Shape : {X_test.shape}  | y_test Churn Rate : {y_test.mean():.2%}")


### [STEP] Step 4.2: Domain-Specific Interaction Features & Safe Ratio Construction
Raw features in isolation often fail to expose business dynamics. We construct domain interaction features:
* `charge_per_tenure_month`: How heavily the customer is charged per month of tenure.
* `monthly_to_total_ratio`: The proportion of total lifetime charges represented by the current monthly plan.

We use our `safe_divide()` utility to guard against zero-division.


In [ ]:
# Step 4.2: Domain-Specific Interaction Features

def construct_domain_features(df: pd.DataFrame) -> pd.DataFrame:
    """Constructs domain-specific ratio and interaction features."""
    df_feat = df.copy()
    
    # 1. Charge per tenure month ratio
    df_feat["charge_per_tenure_month"] = safe_divide(
        df_feat["total_charges"], 
        df_feat["tenure_months"]
    )
    
    # 2. Monthly to total charges ratio
    df_feat["monthly_to_total_ratio"] = safe_divide(
        df_feat["monthly_charges"], 
        df_feat["total_charges"]
    )
    
    return df_feat

# Apply independently to train and test
X_train = construct_domain_features(X_train)
X_test = construct_domain_features(X_test)

print("--- Step 4.2: Domain Features Constructed ---")
print(X_train[["tenure_months", "monthly_charges", "charge_per_tenure_month", "monthly_to_total_ratio"]].head(3))


### [STEP] Step 4.3: Datetime Decomposition & Cyclical Sine/Cosine Transformations
#### [THEORY] Why Cyclical Encoding is Mandatory:
Calendar months range from $1$ to $12$. If fed to a machine learning model as raw integers, month $12$ (December) and month $1$ (January) have a mathematical difference of $|12 - 1| = 11$, when in physical reality they are consecutive months!

We map periodic time features onto the **unit circle** using Trigonometric Sine and Cosine coordinates:
$$x_{\sin} = \sin\left(\frac{2\pi \times t}{T}\right), \quad x_{\cos} = \cos\left(\frac{2\pi \times t}{T}\right)$$
Now, December ($t=12$) and January ($t=1$) are naturally adjacent in Euclidean distance!


In [ ]:
# Step 4.3: Datetime Decomposition & Cyclical Encoding

def extract_datetime_features(df: pd.DataFrame, dt_col: str = "timestamp") -> pd.DataFrame:
    """Extracts calendar sub-components and cyclical trigonometric projections."""
    df_feat = df.copy()
    if dt_col in df_feat.columns:
        dt_series = pd.to_datetime(df_feat[dt_col])
        
        df_feat["join_hour"] = dt_series.dt.hour
        df_feat["join_dayofweek"] = dt_series.dt.dayofweek
        df_feat["join_month"] = dt_series.dt.month
        
        # Cyclical Sine and Cosine Transformations (T = 12 for months, T = 24 for hours)
        df_feat["month_sin"] = np.sin(2 * np.pi * df_feat["join_month"] / 12.0)
        df_feat["month_cos"] = np.cos(2 * np.pi * df_feat["join_month"] / 12.0)
        df_feat["hour_sin"] = np.sin(2 * np.pi * df_feat["join_hour"] / 24.0)
        df_feat["hour_cos"] = np.cos(2 * np.pi * df_feat["join_hour"] / 24.0)
        
        # Drop raw timestamp and raw intermediate non-cyclical calendar markers
        df_feat = df_feat.drop(columns=[dt_col, "join_hour", "join_month"])
    return df_feat

X_train = extract_datetime_features(X_train, "timestamp")
X_test = extract_datetime_features(X_test, "timestamp")

print("--- Step 4.3: Cyclical Features Preview ---")
print(X_train[["month_sin", "month_cos", "hour_sin", "hour_cos"]].head(3))


### [STEP] Step 4.4: Categorical Encoding & Rare Category Management
#### [THEORY] Managing Cardinality:
* **Rare Category Binning**: Infrequent categories ($< 5\%$) are grouped into an `'Other'` bucket to prevent high-variance overfitting and unseen category errors at test time.
* **One-Hot Encoding**: Low-cardinality nominal features are converted into binary columns. We set `drop='first'` to prevent linear collinearity (the "dummy variable trap") and `handle_unknown='ignore'` so unexpected test labels don't crash inference.


In [ ]:
# Step 4.4: Rare Category Binning & Scikit-Learn One-Hot Encoding

class ProductionCategoricalEncoder:
    """Encodes low-cardinality categories with rare grouping and unknown handling."""
    def __init__(self, cat_cols: List[str], min_frequency: float = 0.05):
        self.cat_cols = cat_cols
        self.min_frequency = min_frequency
        self.rare_category_maps = {}
        self.ohe = None
        
    def fit(self, df: pd.DataFrame):
        df_clean = df.copy()
        # 1. Learn rare categories on training partition only
        for col in self.cat_cols:
            if col in df_clean.columns:
                freq = df_clean[col].value_counts(normalize=True)
                rare_labels = freq[freq < self.min_frequency].index.tolist()
                self.rare_category_maps[col] = rare_labels
                df_clean[col] = df_clean[col].replace(rare_labels, "other")
                
        # 2. Fit Scikit-Learn OneHotEncoder
        self.ohe = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
        self.ohe.fit(df_clean[self.cat_cols])
        return self
        
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df_clean = df.copy()
        # Apply learned rare category maps
        for col, rare_labels in self.rare_category_maps.items():
            if col in df_clean.columns:
                df_clean[col] = df_clean[col].replace(rare_labels, "other")
                
        # Transform via fitted OneHotEncoder
        encoded_array = self.ohe.transform(df_clean[self.cat_cols])
        encoded_cols = self.ohe.get_feature_names_out(self.cat_cols)
        encoded_df = pd.DataFrame(encoded_array, columns=encoded_cols, index=df_clean.index)
        
        # Combine non-categorical features with new encoded features
        remaining_cols = [c for c in df_clean.columns if c not in self.cat_cols]
        return pd.concat([df_clean[remaining_cols], encoded_df], axis=1)

# Fit exclusively on X_train, transform both train and test
cat_features = ["contract_type", "payment_method"]
cat_encoder = ProductionCategoricalEncoder(cat_cols=cat_features, min_frequency=0.05)
cat_encoder.fit(X_train)

X_train = cat_encoder.transform(X_train)
X_test = cat_encoder.transform(X_test)

# Drop raw non-predictive ID column
X_train = X_train.drop(columns=["customer_id"])
X_test = X_test.drop(columns=["customer_id"])

print(f"--- Step 4.4: One-Hot Encoding Complete ---")
print(f"Training Features Count: {X_train.shape[1]} columns")


### [STEP] Step 4.5: Low-Variance & Quasi-Constant Feature Elimination
Features with near-zero variance contain virtually identical values for all records. They contribute zero predictive discrimination while inflating computational overhead.
* A feature with $\sigma^2 = 0$ is completely constant.
* A quasi-constant feature (e.g., $98\%$ ones, $2\%$ zeros) has variance $p(1-p) = 0.02 \times 0.98 = 0.0196$.

We use `VarianceThreshold(threshold=0.01)` to automatically drop them.


In [ ]:
# Step 4.5: Low-Variance & Quasi-Constant Feature Elimination

# Intentionally append a constant feature and a quasi-constant feature for demonstration
X_train["dummy_constant"] = 1.0
X_test["dummy_constant"] = 1.0

X_train["dummy_quasi"] = np.random.choice([0, 1], size=len(X_train), p=[0.99, 0.01])
X_test["dummy_quasi"] = np.random.choice([0, 1], size=len(X_test), p=[0.99, 0.01])

# Apply VarianceThreshold fitted on train
var_selector = VarianceThreshold(threshold=0.02)
var_selector.fit(X_train)

retained_columns = X_train.columns[var_selector.get_support()]
dropped_columns = X_train.columns[~var_selector.get_support()].tolist()

X_train = pd.DataFrame(var_selector.transform(X_train), columns=retained_columns, index=X_train.index)
X_test = pd.DataFrame(var_selector.transform(X_test), columns=retained_columns, index=X_test.index)

print("--- Step 4.5: Variance Filtering Complete ---")
print(f"Dropped Low-Variance Columns: {dropped_columns}")
print(f"Retained Features Count: {X_train.shape[1]}")


### [STEP] Step 4.6: Multicollinearity Detection & Redundancy Removal
#### [THEORY] Why Multicollinearity Degrades Models:
When two features are almost perfectly correlated ($|r| > 0.85$ or $0.90$), they provide redundant information. In linear models, logistic regression, and neural networks, multicollinearity inflates parameter standard errors and makes feature coefficients unstable.

We compute the pairwise correlation matrix on training data and drop one feature from each highly collinear pair.


In [ ]:
# Step 4.6: Multicollinearity Detection & Redundancy Removal

def remove_collinear_features(df_train: pd.DataFrame, df_test: pd.DataFrame, threshold: float = 0.85) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    """
    Computes pairwise correlation on training features.
    Drops one feature from any pair with absolute correlation > threshold.
    """
    corr_matrix = df_train.corr().abs()
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    
    to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > threshold)]
    
    df_train_reduced = df_train.drop(columns=to_drop)
    df_test_reduced = df_test.drop(columns=to_drop)
    
    return df_train_reduced, df_test_reduced, to_drop

X_train, X_test, collinear_dropped = remove_collinear_features(X_train, X_test, threshold=0.85)

print("--- Step 4.6: Multicollinearity Filtering Complete ---")
print(f"Dropped Collinear Features: {collinear_dropped}")
print(f"Active Features Remaining : {X_train.shape[1]}")


### [STEP] Step 4.7: Power Transformations (Yeo-Johnson) & Feature Scaling
#### [THEORY] Skewness Correction and Standardization:
* **Yeo-Johnson Power Transformation**: Continuous financial metrics (`total_charges`, `monthly_charges`) typically follow heavy right-skewed lognormal distributions. Yeo-Johnson normalizes the distribution while supporting positive, zero, and negative values (unlike Box-Cox which requires strictly positive values).
* **StandardScaler**: Brings all continuous features to mean $\mu = 0$ and unit variance $\sigma = 1$, enabling equal gradient descent updates across dimensions.


In [ ]:
# Step 4.7: Skewness Correction (Yeo-Johnson) & StandardScaler

# 1. Identify continuous skewed features
skewed_cols = ["total_charges", "monthly_charges"]
skewed_cols = [c for c in skewed_cols if c in X_train.columns]

# Fit PowerTransformer strictly on train, transform test
pt = PowerTransformer(method="yeo-johnson", standardize=False)
X_train[skewed_cols] = pt.fit_transform(X_train[skewed_cols])
X_test[skewed_cols] = pt.transform(X_test[skewed_cols])

# 2. Fit StandardScaler strictly on train, transform test
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print("--- Step 4.7: Power Transformation & Scaling Complete ---")
print(f"X_train_scaled Feature Means (approx 0):\n{X_train_scaled.mean().round(2).head(4)}")
print(f"\nX_train_scaled Feature Stds (approx 1):\n{X_train_scaled.std().round(2).head(4)}")


### [STEP] Step 4.8: Automated Feature Selection & Dimensionality Reduction (PCA)
#### [THEORY] Filter Selection vs Dimensionality Reduction:
* **`SelectKBest` (ANOVA F-Score)**: Evaluates the linear discriminative relationship between each feature and the target variable ($y$), retaining only the top $k$ most informative features.
* **Principal Component Analysis (PCA)**: Projects high-dimensional feature spaces onto orthogonal principal axes, capturing maximal cumulative variance.


In [ ]:
# Step 4.8: Automated Feature Selection & PCA Projection

# 1. Feature Selection via ANOVA F-Classif
k_best = min(8, X_train_scaled.shape[1])
selector = SelectKBest(score_func=f_classif, k=k_best)
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)

selected_features = X_train_scaled.columns[selector.get_support()].tolist()
feature_scores = selector.scores_[selector.get_support()]

print("--- Step 4.8: Top Selected Features (ANOVA F-Classif) ---")
for feat, score in sorted(zip(selected_features, feature_scores), key=lambda x: x[1], reverse=True):
    print(f" - {feat:30} : F-Score = {score:.4f}")

# 2. Dimensionality Reduction via PCA
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_selected)
X_test_pca = pca.transform(X_test_selected)

explained_variance = pca.explained_variance_ratio_
print(f"\nPCA 2-Component Cumulative Variance Captured: {np.sum(explained_variance)*100:.2f}%")
print(f"Component 1 explains: {explained_variance[0]*100:.2f}%")
print(f"Component 2 explains: {explained_variance[1]*100:.2f}%")


### [STEP] Step 4.9: Transformer Serialization & Production Feature Matrix Export
In production MLOps, we must save our fitted encoders, scalers, and selectors (`joblib.dump`) so our live inference API can process new incoming requests using identical transformations.

We persist our model artifacts and export finalized feature matrices for **Stage 5**.


In [ ]:
# Step 4.9: Artifact Serialization & Feature Matrix Export

models_dir = Path("models")
models_dir.mkdir(parents=True, exist_ok=True)

# 1. Serialize Fitted Transformers
joblib.dump(cat_encoder.ohe, models_dir / "fitted_ohe_encoder.pkl")
joblib.dump(cat_encoder.rare_category_maps, models_dir / "rare_category_maps.pkl")
joblib.dump(pt, models_dir / "fitted_power_transformer.pkl")
joblib.dump(scaler, models_dir / "fitted_standard_scaler.pkl")
joblib.dump(selector, models_dir / "fitted_feature_selector.pkl")
joblib.dump(pca, models_dir / "fitted_pca_projector.pkl")

# 2. Export Production-Ready Feature Matrices
final_train_features = pd.DataFrame(X_train_selected, columns=selected_features)
final_train_features["churn_target"] = y_train.values

final_test_features = pd.DataFrame(X_test_selected, columns=selected_features)
final_test_features["churn_target"] = y_test.values

final_train_features.to_csv(processed_data_dir / "train_features_stage4.csv", index=False)
final_test_features.to_csv(processed_data_dir / "test_features_stage4.csv", index=False)

logger.info(f"[OK] Serialized 5 transformer artifacts into '{models_dir}'.")
logger.info(f"[OK] Exported train feature matrix ({final_train_features.shape}) -> {processed_data_dir / 'train_features_stage4.csv'}")
logger.info(f"[OK] Exported test feature matrix  ({final_test_features.shape})  -> {processed_data_dir / 'test_features_stage4.csv'}")

print("--- Final Production Artifacts ---")
for p in models_dir.glob("*.pkl"):
    print(f"[ARTIFACT] Model Artifact: {p.name}")


---
# ====================================================================
# [MASTERCLASS] MASTERCLASS GRADUATION & SUMMARY ROADMAP
# ====================================================================

### [SUMMARY] What You Have Mastered Today:
Congratulations! You have bridged theory and real-world code across the first four foundational stages of the Machine Learning lifecycle:

1. **Stage 1 (Scoping)**:
   * Defined financial cost matrices ($C_{FN} \gg C_{FP}$) to evaluate models by dollar impact, not just accuracy.
   * Enforced GDPR/HIPAA privacy compliance with cryptographic salted SHA-256 PII masking.
   * Established quantitative heuristic baseline benchmarks (`DummyClassifier`).

2. **Stage 2 (Ingestion & Low-Level Hygiene)**:
   * Verified file integrity using zero-byte checks and SHA-256 chunked hashing.
   * Intercepted schema drift and routed invalid records to a Dead-Letter Queue (Quarantine).
   * Downcasted integer and floating-point types, slashing RAM footprint.
   * Implemented out-of-core chunked streaming.

3. **Stage 3 (Preprocessing, Cleaning & EDA)**:
   * Diagnosed MCAR/MAR/MNAR missingness, masked sentinel codes (`-999.0`), added `_is_missing` flags, and applied KNN imputation.
   * Standardized structures with `snake_case`, text trimming, deduplication, and Levenshtein string matching.
   * Capped statistical outliers using Tukey's IQR fences ($1.5 \times IQR$) and Z-Score bounds.
   * Prevented look-ahead leakage in time-series via UTC alignment and forward-fill (`ffill`) resampling.
   * Sanitized floating-point matrices against IEEE 754 infinities and division by zero.
   * Profiled bivariate target interactions.

4. **Stage 4 (Feature Engineering, Transformation & Selection)**:
   * Enforced strict train/test fit-transform separation to prevent catastrophic data leakage.
   * Built domain interaction ratios and trigonometric cyclical Sine/Cosine datetime features.
   * Grouped rare categories and applied one-hot encoding with unknown label tolerance.
   * Filtered quasi-constant features with `VarianceThreshold` and eliminated multicollinearity ($|r| > 0.85$).
   * Normalized skewed continuous features using Yeo-Johnson power transformations and scaled features with `StandardScaler`.
   * Selected optimal feature subsets with ANOVA F-Classif and projected dimensions via PCA.
   * Serialized production artifacts into `.pkl` binaries for downstream serving.

---

### [STAGE] What's Ahead in Stages 5 to 9:
With your clean, sanitized, and serialized feature matrices (`train_features_stage4.csv` and `test_features_stage4.csv`), you are now fully equipped to proceed to:
* **Stage 5**: Model Training, Cross-Validation & Experimentation (Logistic Regression, Random Forests, XGBoost).
* **Stage 6**: Hyperparameter Optimization (Optuna, Bayesian Search).
* **Stage 7**: Model Evaluation & Explainability (SHAP, LIME, ROC-AUC, Cost Curve optimization).
* **Stage 8**: Model Serialization & MLOps Deployment (FastAPI, Docker, Triton).
* **Stage 9**: Production Monitoring, Concept Drift & Automated Retraining (Evidently AI, Prometheus).

*Keep this notebook as your trusted reference pipeline whenever you start any new machine learning engineering project!*
